# 02 Shared Pipeline and Evaluation

This notebook documents the shared Phase 1 foundation used by every model in the project. It explains how raw Myket interactions are converted into train/validation/test splits and how all variants are evaluated under the same top-K ranking protocol.

## What this notebook establishes
- `phase1/preprocess.py` performs cleaning, filtering, id encoding, temporal splitting, and artifact generation.
- `phase1/common.py` defines the shared loading utilities, train matrix construction, seen-item masking, and metric computation.
- All variants consume the same processed files in `phase1/processed/` and report the same metrics: `Precision@10`, `Recall@10`, and `NDCG@10`.
- The evaluation filters to users with both training history and future validation/test interactions, then masks items already seen in the training split.

The cells below expose the processed schema, split sizes, temporal ranges, eligible-user counts, and mapping files that make the shared evaluation reproducible.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

from analysis.shared_utils import (
    PROCESSED_DIR,
    eligible_user_summary,
    load_dataset_stats,
    load_processed_splits,
    load_schema,
    split_time_ranges,
)

stats = load_dataset_stats()
schema = load_schema()
train_df, val_df, test_df = load_processed_splits()
time_ranges = split_time_ranges()
val_summary = eligible_user_summary("val", k=10)
test_summary = eligible_user_summary("test", k=10)

display(Markdown("## Dataset Stats Written by `phase1/preprocess.py`"))
display(pd.DataFrame([stats]))

print("Processed schema columns")
display(pd.DataFrame({"processed_columns": schema["processed"]["columns"]}))

print("Pipeline output files")
display(pd.DataFrame({"output_file": schema["outputs"]}))

print("Split chronology and cardinalities")
display(time_ranges)

print("Validation evaluation cohort")
display(val_summary)

print("Test evaluation cohort")
display(test_summary)

print("Processed directory:", PROCESSED_DIR)

## Dataset Stats Written by `phase1/preprocess.py`

,input_csv,raw_rows,rows_after_cleaning,num_users,num_items,sparsity,split_rows,filter_thresholds
0,myket-android-application-market-dataset/myket...,694121,693755,10000,7863,0.991177,"{'train': 555004, 'val': 69375, 'test': 69376}","{'min_user_interactions': 5, 'min_item_interac..."


Processed schema columns


,processed_columns
0,user_id
1,item_id
2,timestamp
3,timestamp_dt


Pipeline output files


...

Validation evaluation cohort


,eval_split,users_in_eval_split,eligible_users,excluded_no_train_history,users_with_truth_lt_k,min_candidates_after_train_mask,median_truth_size,max_truth_size
0,val,7571,7568,3,5366,7244,6.0,111


Test evaluation cohort


,eval_split,users_in_eval_split,eligible_users,excluded_no_train_history,users_with_truth_lt_k,min_candidates_after_train_mask,median_truth_size,max_truth_size
0,test,7084,7080,4,4909,7244,6.0,167


Processed directory: /Users/ritwikreddy/Desktop/2nd semster/Recommender Systems/final_project/recommender_systems/phase1/processed


In [2]:
user_map = pd.read_csv(PROCESSED_DIR / "user_mapping.csv")
item_map = pd.read_csv(PROCESSED_DIR / "item_mapping.csv")

split_overview = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "users": train_df["user_id"].nunique(), "items": train_df["item_id"].nunique()},
        {"split": "val", "rows": len(val_df), "users": val_df["user_id"].nunique(), "items": val_df["item_id"].nunique()},
        {"split": "test", "rows": len(test_df), "users": test_df["user_id"].nunique(), "items": test_df["item_id"].nunique()},
    ]
)

display(Markdown("## Encoded ID Mapping Samples"))
display(user_map.head())
display(item_map.head())

print("Split overview")
display(split_overview)

print(
    "Evaluation contract summary:\n"
    "- temporal global 80/10/10 split after sorting by timestamp\n"
    "- users must appear in both train and eval split to be scored\n"
    "- ranking masks items seen in train only\n"
    "- Precision uses fixed K, Recall uses the user's truth size, and NDCG uses min(|truth|, K)"
)

## Encoded ID Mapping Samples

,original_user_id,user_id
0,-100077120,0
1,-1001608536,1
2,-1001801844,2
3,-100223082,3
4,-1002720552,4


,original_item_id,item_id
0,Ahmad.spongebobstar2,0
1,Awesome.Hide2,1
2,COM.ANTIVIROD.GHAVYS,2
3,COM.LIMIT.DAY,3
4,Call.block,4


Split overview


,split,rows,users,items
0,train,555004,9996,7823
1,val,69375,7571,6606
2,test,69376,7084,6599


Evaluation contract summary:
- temporal global 80/10/10 split after sorting by timestamp
- users must appear in both train and eval split to be scored
- ranking masks items seen in train only
- Precision uses fixed K, Recall uses the user's truth size, and NDCG uses min(|truth|, K)
